# KAL pilot 2 on a cloud GPU (docs/privileged_teacher_case.md, section 6d)

Trains every pilot-2 arm for each seed on the GPU, evaluates it on every cohort present, and saves each finished seed as
`kal_pilot2_seed_<seed>.zip` straight away. A session that stops costs only the seeds in progress: run the notebook again
(with the earlier output added as input on Kaggle, or from the same Drive folder on Colab) and finished work is skipped.
The code and settings are the laptop's; only the device differs. Steps: `cloud/README.md`.

In [ ]:
# 1. Find the uploaded package and set up the working copy (code writable, data linked read-only)
import glob, os, shutil, sys, zipfile
KAGGLE = os.path.exists('/kaggle/input')
if not KAGGLE:
    from google.colab import drive
    drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/kal_pilot2'          # Colab: upload the zips here; results are written here too
WORK = '/kaggle/working' if KAGGLE else '/content'
RESULTS = WORK if KAGGLE else DRIVE
FYP = os.path.join(WORK, 'fyp')
PA = os.path.join(FYP, 'out', 'm1', 'privileged_ablation')
os.makedirs(RESULTS, exist_ok=True)

def find(pattern):
    root = '/kaggle/input' if KAGGLE else DRIVE
    return sorted(glob.glob(os.path.join(root, '**', pattern), recursive=True))

def link_tree(src_fyp):
    # code and small files copied; every cache folder and every scene metadata file linked
    def ignore(d, names):
        d = d.replace(os.sep, '/')
        return [n for n in names if (n == 'cache' and '/out/m1/' in d + '/') or (n == 'meta' and '/out/m1/dataset' in d)]
    shutil.copytree(src_fyp, FYP, ignore=ignore, dirs_exist_ok=True)
    for c in glob.glob(os.path.join(src_fyp, 'out', 'm1', '**', 'cache'), recursive=True):
        dst = os.path.join(FYP, os.path.relpath(c, src_fyp))
        if not os.path.exists(dst):
            os.symlink(c, dst)
    for m in glob.glob(os.path.join(src_fyp, 'out', 'm1', 'dataset*', 'meta', '*.json')):
        dst = os.path.join(FYP, os.path.relpath(m, src_fyp))
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        if not os.path.exists(dst):
            os.symlink(m, dst)

# every uploaded package tree (Kaggle unpacks an uploaded zip into a folder named after the dataset); the one with the
# code first. Zips that were not unpacked (Colab, or Kaggle with a zip inside a folder) are extracted instead.
trees = sorted({p.replace(os.sep, '/').split('/fyp/')[0] + '/fyp' for p in find('fyp/out/m1/privileged_ablation/manifest.json')
                + find('fyp/out/m1/privileged_ablation/*/manifest.json')})
trees.sort(key=lambda t: not os.path.exists(os.path.join(t, 'tools', 'm1_det2.py')))
for t in trees:
    if os.path.abspath(t) != os.path.abspath(FYP):
        link_tree(t)
if not trees:                                             # never extract over linked read-only folders
    for z in find('kal_pilot2_train.zip') + find('kal_pilot2_cohorts.zip'):
        zipfile.ZipFile(z).extractall(WORK)
if not os.path.exists(os.path.join(FYP, 'tools', 'm1_det2.py')):
    raise SystemExit('kal_pilot2_train.zip not found: upload it first (cloud/README.md)')
print('working copy:', FYP, '| cohorts:', [d for d in ('thermal_low_heat', 'thermal_electric_like', 'fresh')
                                           if os.path.exists(os.path.join(PA, d, 'cache'))])

In [ ]:
# 2. Resume: unpack earlier sessions' seeds (Kaggle: add the earlier version's output as an input first)
for z in find('kal_pilot2_seed_*.zip') + find('kal_pilot2_results_*.zip'):
    zipfile.ZipFile(z).extractall(PA)
    print('resumed from', z)

In [ ]:
# 3. The GPU
import torch
print(torch.__version__, 'CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
# 4. Train, evaluate and save. Section 6d: at least seeds 101-120; after the laptop's interim, set SEEDS to the number in
#    out/m1/privileged_ablation/pilot2_power.json (for example '101-130') and run again: finished seeds are skipped.
SEEDS, SENSOR_SEEDS, WORKERS = '101-120', '101-115', 3
!cd {FYP} && KAL_DEVICE=cuda KAL_THREADS=1 KAL_RESULTS_DIR={RESULTS} python -u tools/cloud_pilot2.py run --seeds {SEEDS} --sensor-seeds {SENSOR_SEEDS} --workers {WORKERS}

In [ ]:
# 5. What to download: every kal_pilot2_seed_*.zip and the latest kal_pilot2_results_*.zip
print('
'.join(sorted(glob.glob(os.path.join(RESULTS, 'kal_pilot2_*.zip')))))